管道提示词可以将多个提示组合在一起。当您想要重复使用部分提示时，这会很有用。这可以通过 PipelinePrompt 来完成。

PipelinePrompt 由两个主要部分组成：
- 最终提示：返回的最终提示
- 管道提示：元组列表，由字符串名称和提示模板组成。每个提示模板将被格式化，然后作为具有相同名称的变量传递到未来的提示模板。

In [1]:
from langchain.prompts.pipeline import PipelinePromptTemplate
from langchain.prompts.prompt import PromptTemplate

In [2]:
full_template = """{introduction}

{example}

{start}"""
full_prompt = PromptTemplate.from_template(full_template)

In [3]:
introduction_template = """你正在冒充{person}。"""
introduction_prompt = PromptTemplate.from_template(introduction_template)

In [4]:
example_template = """
下面是一个交互示例：

Q：{example_q}
A：{example_a}"""
example_prompt = PromptTemplate.from_template(example_template)

In [5]:
start_template = """现在正式开始！

Q：{input}
A："""
start_prompt = PromptTemplate.from_template(start_template)

In [6]:
input_prompts = [
    ("introduction", introduction_prompt),
    ("example", example_prompt),
    ("start", start_prompt),
]
pipeline_prompt = PipelinePromptTemplate(
    final_prompt=full_prompt, pipeline_prompts=input_prompts
)

C:\Users\jerry\AppData\Local\Temp\ipykernel_18892\1772176540.py:6: LangChainDeprecationWarning: This class is deprecated in favor of chaining individual prompts together.
  pipeline_prompt = PipelinePromptTemplate(


In [7]:
pipeline_prompt.input_variables

['person', 'input', 'example_a', 'example_q']

In [9]:
print(
    pipeline_prompt.format(
        person="Elon Musk",
        example_q="你最喜欢什么车？",
        example_a="Tesla",
        input="您最喜欢的社交媒体网站是什么?",
    )
)

你正在冒充Elon Musk。


下面是一个交互示例：

Q：你最喜欢什么车？
A：Tesla

现在正式开始！

Q：您最喜欢的社交媒体网站是什么?
A：


In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
chat = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.7,
)

output_parser = StrOutputParser()

chain = pipeline_prompt | chat | output_parser

chain.invoke({
    "input":"您最喜欢的社交媒体网站是什么",
    "person":"Elon Musk",
    "example_q":"你最喜欢什么车？",
    "example_a":"Tesla",
})

'<think>\n好的，用户让我扮演埃隆·马斯克，并且现在问的是“您最喜欢的社交媒体网站是什么”。首先，我需要回忆一下马斯克常用的社交平台。他主要活跃在Twitter（X）上，这是他的主要发声渠道，尤其是在推动特斯拉、SpaceX和PayPal等方面。\n\n接下来，我应该考虑用户可能的意图。他们可能想了解马斯克的个人偏好，或者测试我的角色扮演能力。作为AI助手，我需要保持一致性，同时给出符合马斯克风格的回答。\n\n然后，我要确保回答准确且有个性。马斯克经常在Twitter上发布推文，所以正确的答案应该是Twitter（X）。不过，要注意到X已经更名为Twitter，但用户可能还是用旧名称来称呼它。此外，是否需要提到其他平台呢？比如Instagram或YouTube？不过马斯克主要活动在Twitter，所以应该以X为主。\n\n另外，回答要简洁有力，符合马斯克的风格，带点自信和直接。可能还需要加入一些个人特色，比如提到他的公司或者项目，但不要过于冗长。例如，“Twitter（X）是我在全球范围内沟通、发布信息和推动变革的主要平台。”这样既准确又符合角色。\n\n最后，检查是否有其他可能的回答，确保没有遗漏重要信息。确认无误后，给出最终答案。\n</think>\n\nTwitter（X）是我在全球范围内沟通、发布信息和推动变革的主要平台。它让我能直接与世界各地的人互动，同时分享特斯拉、SpaceX和PayPal的进展。当然，我也在Instagram和YouTube上活跃，但核心还是Twitter——毕竟，这是我的“战场”。'